In [ ]:
!pip -q install tensorflow pandas numpy matplotlib seaborn opencv-python pillow

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from PIL import Image
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

classes = class_names  # Used by the later analysis sections\n\ndataset_path = "/content/cifar10_audit/train"
os.makedirs(dataset_path, exist_ok=True)

# Create class folders and save a balanced subset of training images.
max_images_per_class = 500
saved_counts = {name: 0 for name in class_names}

for image, label_array in zip(x_train, y_train):
    label = int(label_array[0])
    class_name = class_names[label]

    if saved_counts[class_name] >= max_images_per_class:
        continue

    class_folder = os.path.join(dataset_path, class_name)
    os.makedirs(class_folder, exist_ok=True)

    image_path = os.path.join(
        class_folder, f"{class_name}_{saved_counts[class_name]:04d}.png"
    )
    Image.fromarray(image).save(image_path)
    saved_counts[class_name] += 1

    if all(count >= max_images_per_class for count in saved_counts.values()):
        break

print("CIFAR-10 dataset prepared successfully!")
print("Dataset location:", dataset_path)
print("Images saved:", sum(saved_counts.values()))
print("Images per class:", saved_counts)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 534s 3us/step


NameError: name 'dataset_path' is not defined

In [ ]:
plt.figure(figsize=(14, 8))
image_count = 0

for class_name in classes:
    folder = os.path.join(dataset_path, class_name)
    images = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    if not images:
        continue

    image_path = os.path.join(folder, images[0])
    img = Image.open(image_path).convert("RGB")

    plt.subplot(2, 5, image_count + 1)
    plt.imshow(img)
    plt.title(class_name)
    plt.axis("off")
    image_count += 1

    if image_count == 10:
        break

plt.suptitle("CIFAR-10 Sample Images", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
class_counts = {}

for class_name in classes:
    folder = os.path.join(dataset_path, class_name)
    images = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    class_counts[class_name] = len(images)

print("Class counts:", class_counts)

plt.figure(figsize=(12, 5))
sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()))
plt.title("CIFAR-10 Class Distribution")
plt.xlabel("Classes")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.show()

In [ ]:
quality_data = []

for class_name in classes:
    folder = os.path.join(dataset_path, class_name)
    images = sorted(os.listdir(folder))

    for file in images:
        if not file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        path = os.path.join(folder, file)
        img = cv2.imread(path)

        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
            height, width = img.shape[:2]

            quality_data.append({
                "Class": class_name,
                "Image": file,
                "Width": width,
                "Height": height,
                "Sharpness": sharpness
            })

quality_df = pd.DataFrame(quality_data)

display(quality_df.head(10))
print("Images analysed:", len(quality_df))
print("Average sharpness:", quality_df["Sharpness"].mean())

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(quality_df["Sharpness"], bins=30, kde=True)
plt.title("CIFAR-10 Image Sharpness Distribution")
plt.xlabel("Laplacian Variance")
plt.ylabel("Frequency")
plt.show()

In [ ]:
print("Image Resolution Analysis")
print("Average Width:", quality_df["Width"].mean())
print("Average Height:", quality_df["Height"].mean())
print("\\nMinimum Width:", quality_df["Width"].min())
print("Maximum Width:", quality_df["Width"].max())
print("\\nMinimum Height:", quality_df["Height"].min())
print("Maximum Height:", quality_df["Height"].max())

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(quality_df["Width"], quality_df["Height"], alpha=0.5)
plt.title("CIFAR-10 Image Resolution Distribution")
plt.xlabel("Image Width")
plt.ylabel("Image Height")
plt.show()

In [ ]:
valid = 0
invalid = 0
unreadable = 0

for class_name in classes:
    folder = os.path.join(dataset_path, class_name)

    for file in os.listdir(folder):
        if not file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        path = os.path.join(folder, file)

        try:
            with Image.open(path) as img:
                img.verify()
                if img.format in ["JPEG", "PNG"]:
                    valid += 1
                else:
                    invalid += 1
        except Exception:
            unreadable += 1

print("Image Validation Results")
print("Valid Images:", valid)
print("Invalid Images:", invalid)
print("Unreadable Images:", unreadable)

In [ ]:
print("\\n====================================")
print("       FINAL DATASET AUDIT")
print("====================================")

print("Dataset: CIFAR-10 (balanced training subset)")
print("Total Classes:", len(class_counts))
print("Total Images:", sum(class_counts.values()))

print("\\nCLASS DISTRIBUTION")
for name, count in class_counts.items():
    print(name, ":", count)

print("\\nIMAGE QUALITY")
print("Average Sharpness:", round(quality_df["Sharpness"].mean(), 2))

print("\\nIMAGE RESOLUTION")
print("Average Width:", round(quality_df["Width"].mean(), 2))
print("Average Height:", round(quality_df["Height"].mean(), 2))

print("\\nIMAGE VALIDATION")
print("Valid Images:", valid)
print("Invalid Images:", invalid)
print("Unreadable Images:", unreadable)

print("\\n====================================")
print("DATASET AUDIT COMPLETED!")
print("====================================")